# Lesson 8: One self-attention head

**Why:** in Lesson 7 every earlier character counted **equally**. But when finishing the word "current" in "alternating current", the letters of "alternating" matter much more than a space or a comma. **Attention** lets each character decide **how much** each earlier character matters to it.

**The idea, as a search:** every character produces three things:

| | Question it answers | Analogy |
|---|---|---|
| **Query (q)** | "What am I looking for?" | what you type into a search box |
| **Key (k)** | "What do I contain?" | the title of each search result |
| **Value (v)** | "What will I share if you pick me?" | the content of the result |

A character's query is compared with every earlier character's key (a dot product, the "how well do they line up" measure from Lesson 1). Good matches get high scores → mask the future → softmax → a weighted mix of the **values**.

```
x (B,T,C) ─┬─ query ─► q (B,T,hs) ─┐
           ├─ key   ─► k (B,T,hs) ─┴─► q @ kᵀ / √hs ─► (B,T,T) ─► mask ─► softmax ─► wei ─┐
           └─ value ─► v (B,T,hs) ─────────────────────────────────────────────────────── wei @ v ─► (B,T,hs)
```

## 1. Setup

Same data as before, with two changes: chunks are now **32** characters long (attention can actually use a longer context), and **C = 32** numbers describe each character.

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(1337)

text = Path("../data/input.txt").read_text(encoding="utf-8")
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

batch_size, block_size = 32, 32   # B, T
n_embd = 32                       # C: numbers per character
head_size = 32                    # hs: size of q, k, v

def get_batch(split):
    source = train_data if split == "train" else val_data
    starts = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in starts])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in starts])
    return x, y

@torch.no_grad()
def estimate_loss(model, eval_batches=100):
    model.eval()
    result = {}
    for split in ["train", "val"]:
        losses = [model(*get_batch(split))[1].item() for _ in range(eval_batches)]
        result[split] = sum(losses) / len(losses)
    model.train()
    return result

print("ready")

## 2. One head, step by step, with shapes

On random data first, so we can follow the shapes. `nn.Linear(C, hs, bias=False)` is just a learnable weight grid (Lesson 2): it turns each character's 32 numbers into its query, key or value.

In [ ]:
B, T, C = 1, 8, 32
x = torch.randn(B, T, C)

query = nn.Linear(C, head_size, bias=False)
key   = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

q = query(x)                                     # (B, T, hs)  what each char is looking for
k = key(x)                                       # (B, T, hs)  what each char contains
wei = q @ k.transpose(-2, -1) * head_size**-0.5  # (B, T, T)   how well each pair matches
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))  # block the future (Lesson 7)
wei = F.softmax(wei, dim=-1)                     # each row adds up to 1
v = value(x)                                     # (B, T, hs)  what each char shares
out = wei @ v                                    # (B, T, hs)

print("q, k, v:", q.shape)
print("wei:    ", wei.shape)
print("out:    ", out.shape)
print("\nweights for the last position (it may look at all 8):")
print(wei[0, -1].detach().numpy().round(2))

## 3. Why divide by √hs?

A dot product adds up **hs** multiplications, so with hs = 32 the raw scores come out large and spread wide. Softmax turns large, spread-out scores into an almost **one-hot** result: all attention on one character, as in Lesson 3's "exaggerates differences". A fresh model would then focus on one random character and learn slowly.

Dividing by √hs brings the spread back to about 1, keeping attention **soft** at the start so it can learn where to look.

In [ ]:
q_ = torch.randn(1000, head_size)
k_ = torch.randn(1000, head_size)
raw = (q_ * k_).sum(dim=-1)                 # 1000 dot products
print(f"spread (variance) without scaling: {raw.var().item():.1f}")
print(f"spread (variance) with scaling:    {(raw * head_size**-0.5).var().item():.2f}")

scores = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print("\nsoftmax of normal scores:    ", F.softmax(scores, -1).numpy().round(2))
print("softmax of scores x 8 (big): ", F.softmax(scores * 8, -1).numpy().round(2))

## 4. Plug it into a language model

Two new pieces:
- **`Head`** packs section 2 into a class. (`register_buffer` stores the triangle with the model; it's not learned.)
- **Position embeddings:** attention is a weighted mix, so by itself it doesn't know **where** each character sits. "Tesla" and "alseT" would look the same to it. So we give each position (0–31) its own learnable vector, and **add** it to the character's vector, just like Lesson 1's broadcasting: (B, T, C) + (T, C).

```
idx (B,T) ─► token emb (B,T,C) + position emb (T,C) ─► Head ─► (B,T,hs) ─► Linear ─► logits (B,T,V)
```

**Predict:** the bigram reached a validation loss of **2.41**. Where will this model end up after the same 5,000 steps: about **2.3**, **2.0** or **1.5**?

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.query(x), self.key(x), self.value(x)            # (B, T, hs) each
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5            # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        self.last_wei = wei.detach()                                   # saved for plotting
        return wei @ v                                                 # (B, T, hs)


class AttentionLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(V, n_embd)           # what each character is
        self.pos_emb = nn.Embedding(block_size, n_embd)  # where it sits
        self.head = Head(head_size)
        self.lm_head = nn.Linear(head_size, V)           # back to 85 scores

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T))   # (B, T, C)
        x = self.head(x)                                        # (B, T, hs)
        logits = self.lm_head(x)                                # (B, T, V)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]                     # can only see 32 back
            logits, _ = self(idx_cond)
            probs = F.softmax(logits[:, -1, :], dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

model = AttentionLanguageModel()
print("parameters:", sum(p.numel() for p in model.parameters()))

## 5. Train it (about 1 minute)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(5001):
    if step % 500 == 0:
        l = estimate_loss(model)
        print(f"step {step:5d}:  train {l['train']:.3f}   val {l['val']:.3f}")
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("\n--- attention model writes ---")
print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long), 400)[0].tolist()))

## 6. Where does it look?

We feed in a phrase in Tesla's style and plot the learned attention weights. **Row** = the character doing the looking; **column** = the character it looks at. Darker = more attention.

In [ ]:
phrase = "the alternating current motor"
idx = torch.tensor([encode(phrase)])
model.eval()
with torch.no_grad():
    model(idx)
model.train()

w = model.head.last_wei[0]
labels = ["␣" if c == " " else c for c in phrase]
plt.figure(figsize=(7, 7))
plt.imshow(w, cmap="Blues")
plt.xticks(range(len(labels)), labels); plt.yticks(range(len(labels)), labels)
plt.xlabel("looks at"); plt.ylabel("character predicting what comes next")
plt.title("Learned attention (one head)"); plt.show()

## 7. Final check

In [ ]:
final = estimate_loss(model, eval_batches=200)
print(f"attention model val loss: {final['val']:.3f}   (bigram was 2.41)")
assert final["val"] < 2.41, "attention should beat the bigram"
print("Lesson 8 complete. Your model now uses context!")

## Wrap-up question

In your own words (or with your own analogy), what are the **query**, **key** and **value**, and why does the model need **position embeddings**?